# CE893 project notebook: LoRA vs prompt-engineering harness for a text-to-Cube agent

This notebook reproduces **every number, table and figure in the report** from the evaluation
outputs stored in this repository, and documents where the data comes from.

* Part 1–5 run anywhere in a few seconds (Python only, no GPU, no Cube, no Ollama): they read the
  stored per-question outputs in `results/`.
* Part 6 shows the commands that **produced** those outputs (data preparation, harness runs, LoRA
  training, export). They need BIRD, Node.js (Cube), Ollama and a 16 GB GPU, so they are switched off
  by default (`RUN_PIPELINE = False`).

Run from the repository root or from `notebook/`: `pip install -r requirements.txt pandas matplotlib sqlglot scipy`.

## 1. Data sources

| What | Source | Licence / terms | Where in this repo |
|---|---|---|---|
| Questions, hints, gold SQL, SQLite databases | BIRD benchmark, Li et al., NeurIPS 2023 — https://bird-bench.github.io/ (`dev.zip`, `train.zip`) | CC BY-SA 4.0 | downloaded by `scripts/download_bird.py` (not committed: ~1.5 GB) |
| Test split (332 q: formula_1 174, student_club 158) | BIRD dev | CC BY-SA 4.0 | `splits/test_ids.json` |
| Harness tuning split (superhero, toxicology, california_schools) | BIRD dev | CC BY-SA 4.0 | `splits/tune_ids.json` (120-q subset); all 363 q used for final tuning comparison |
| LoRA training pairs (3,531 verified question → Cube query) | BIRD train, translated by `harness/sql2cube.py`, verified by execution | CC BY-SA 4.0 (derived) | `training_data/verified.jsonl`, schemas in `training_data/schemas/` |
| Synthetic schema questions (noschema arm only) | generated by `scripts/synth_schema_qa.py` from the two test schemas; no BIRD dev question/hint/SQL used | — | regenerate with the script |
| Cube models | generated from BIRD schema + column descriptions by `scripts/gen_cube_models.py` | — | `cube/model/*.yml` |
| Base models | `Qwen/Qwen2.5-Coder-7B-Instruct`, `-14B-Instruct` (Hugging Face); Ollama `qwen2.5-coder:7b/14b` Q4_K_M | Apache 2.0 | downloaded at run time |
| **Evaluation outputs used in the report** | one JSON line per test question per arm: question, gold SQL, predicted Cube query, strict/relaxed correctness, steps, tokens, latency, full tool trace | — | `results/runs/*.jsonl.gz` |
| **Training logs** | Hugging Face Trainer log history + eval loss before training | — | `results/training/*.json` |

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "report"))
import pandas as pd
import analysis as A
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 80)
print("runs:", sorted(p.name for p in A.RUNS.glob("*.jsonl.gz")))
print("training logs:", sorted(p.name for p in A.TRAINING.glob("*.json")))

In [ ]:
# what one stored output record looks like
r = A.test_run("lora_7b").iloc[1]
{k: (v if k != "trace" else f"[{len(v)} trace steps]") for k, v in r.items() if k not in ("has_join", "called_search_values", "n_tool_calls")}

In [ ]:
# the frozen test split and the training data
test_ids = json.loads((ROOT / "splits/test_ids.json").read_text())
train = pd.read_json(ROOT / "training_data/verified.jsonl", lines=True)
print(len(test_ids), "test questions;", len(train), "verified training pairs on", train.db_id.nunique(), "databases;",
      int((train.searches.str.len() > 0).sum()), "with a search_values step")
train[["db_id", "question", "sql", "cube_query"]].head(3)

## 2. Main results (Table 5 and Fig. 7)

Every reported test run hides BIRD's hint from the model (`--no-evidence`). Runs with the hint are kept under `hint_*` names for reference (section 2b).

In [ ]:
t = A.summary_table()
t[["run", "n", "strict", "relaxed", "prompt_tokens", "steps", "latency_s", "latency_median_s", "latency_p90_s",
   "tokens_per_s", "search_values_pct"]].round(1)

In [ ]:
pairs = [("harness_7b", "lora_7b"), ("harness_14b", "lora_14b"), ("lora_7b", "lora_14b"), ("harness_7b", "harness_14b"),
         ("harness_7b", "lean_7b"), ("harness_14b", "lean_14b"), ("lora_7b", "lean_7b"), ("lora_14b", "lean_14b"),
         ("lora_14b", "noschema_lean_14b"), ("lora_14b", "noschema_14b")]
pairs = [(a, b) for a, b in pairs if A.available([a, b]) == [a, b]]
pd.DataFrame([A.mcnemar(a, b) for a, b in pairs]).assign(p=lambda d: d.p.map("{:.2g}".format))

In [ ]:
rows = []
for run in ["harness_7b", "lora_7b", "harness_14b", "lora_14b"]:
    for by in ["db_id", "has_join", "difficulty"]:
        for k, v in A.breakdown(run, by).items():
            rows.append({"run": run, "group": f"{by}={k}", "strict": round(v, 1)})
pd.DataFrame(rows).pivot(index="group", columns="run", values="strict")

## 2b. Tool use (Table 6), and the same arms with BIRD's hint (for reference)

In [ ]:
A.tool_table(A.MAIN_ARMS).round(2)

In [ ]:
hint = ["hint_" + k for k in A.MAIN_ARMS]
pd.concat([A.summary_table(A.MAIN_ARMS), A.summary_table(hint)])[["run", "strict", "relaxed", "steps", "latency_s"]].round(1)

In [ ]:
pd.DataFrame([A.mcnemar("hint_" + k, k) for k in A.MAIN_ARMS]).assign(p=lambda d: d.p.map("{:.2g}".format))

## 3. Prompt versions and training progress (Fig. 4 and Fig. 6)

In [ ]:
pd.DataFrame({**{f"test {k}": A.summary(A.test_run(k)) for k in ["hint_harness_v1_7b", "hint_harness_v2_7b", "hint_harness_7b"]},
              **{f"tune120 {k}": A.summary(A.load_run(v)) for k, v in A.TUNE_RUNS.items()},
              **{f"tune363 {k}": A.summary(A.load_run(v)) for k, v in A.TUNE363_RUNS.items()}}).T[["n", "strict", "relaxed", "prompt_tokens", "steps"]].round(1)

In [ ]:
pd.DataFrame([{**{k: A.training_log(n)["config"][k] for k in ["base", "max_len", "train_examples", "val_examples"]},
               "run": n, "hours": round(A.training_log(n)["runtime_h"], 2),
               "val_loss": [round(x, 3) for x in A.training_log(n)["eval"]["eval_loss"]]} for n in A.TRAIN_LOGS]).set_index("run")

## 4. Errors and real outputs (Fig. 9 and Fig. 10)

Each wrong answer gets one rule-based category (`analysis.error_category`, rules applied in order). The rules compare the predicted Cube query with the gold SQL; they describe where the query differs, not why the model erred.

In [ ]:
pd.DataFrame({r: A.error_counts(r) for r in A.available(["harness_7b", "lora_7b", "harness_14b", "lora_14b", "lean_14b", "noschema_14b"])})

In [ ]:
h, l = A.test_run("harness_7b"), A.test_run("lora_7b")
for qid in [923, 1314]:
    print(f"Q{qid}: {h.loc[qid, 'question']}\nGOLD: {h.loc[qid, 'gold_sql']}")
    for name, df in [("harness only", h), ("LoRA", l)]:
        print(f"--- 7B {name}: {'correct' if df.loc[qid, 'correct'] else 'wrong'} ({A.error_category(df.loc[qid])})")
        print(A.render_trace(df.loc[qid], 160))
    print()

In [ ]:
# a random sample of LoRA errors, so the report does not show only chosen cases
wrong = l[~l.correct].sample(5, random_state=0)
for qid, row in wrong.iterrows():
    print(f"Q{qid} [{A.error_category(row)}] {row.question}\n  pred: {json.dumps(row.pred_query)[:200]}\n  gold: {row.gold_sql[:200]}\n")

## 5. Regenerate all report figures

In [ ]:
import subprocess
print(subprocess.run([sys.executable, str(ROOT / "report/make_figures.py")], cwd=ROOT / "report", capture_output=True, text=True).stdout)
from IPython.display import Image, display
for f in ["fig07_main", "fig06_training", "fig08_offload", "fig10_errors"]:
    display(Image(filename=str(ROOT / f"report/figures/{f}.png"), width=760))

## 6. How the stored outputs were produced (optional, long-running)

These cells document the full pipeline. Set `RUN_PIPELINE = True` only on a machine with BIRD downloaded,
Cube running on :4000 (`cd cube && npm install && npm run dev`), Ollama, and a 16 GB GPU for training.
Runtimes on an RTX 5070 Ti: one 332-question eval ≈ 10–25 min; 7B LoRA ≈ 5.3 h, 14B ≈ 9 h.

In [ ]:
RUN_PIPELINE = False
STEPS = [
    # data
    "python scripts/download_bird.py",
    "python scripts/build_duckdb.py --bird-dir data/bird --out data/bird.duckdb --db formula_1 --db student_club --db superhero --db toxicology --db california_schools",
    "python scripts/gen_cube_models.py --bird-dir data/bird --out cube/model --db formula_1 --db student_club --db superhero --db toxicology --db california_schools",
    "python scripts/build_training_data.py --bird-dir data/bird_train --out data/training   # needs train.zip; already in training_data/",
    "python scripts/make_sft.py --data training_data --out data/sft",
    # baseline arms
    "python -m harness.evaluate --db formula_1 --db student_club --ids-file splits/test_ids.json --no-evidence --model ollama_chat/qwen2.5-coder:7b --setup harness-v4.2-7b-nohint --out runs/test_nohint_harness_7b.jsonl",
    "python -m harness.evaluate --db formula_1 --db student_club --ids-file splits/test_ids.json --no-evidence --model ollama_chat/qwen2.5-coder:14b --setup harness-v4.2-14b-nohint --out runs/test_nohint_harness_14b.jsonl",
    # LoRA arm (7B; same with 14B and --max-len 4096)
    "python scripts/train_lora.py --base Qwen/Qwen2.5-Coder-7B-Instruct --out lora/qwen7b",
    "python scripts/export_ollama.py --base Qwen/Qwen2.5-Coder-7B-Instruct --adapter lora/qwen7b/final --llama-cpp ../llama.cpp --quantize ../llama.cpp/build/bin/llama-quantize --ollama-base qwen2.5-coder:7b --name qwen7b-cube-lora",
    "python -m harness.evaluate --db formula_1 --db student_club --ids-file splits/test_ids.json --no-evidence --model ollama_chat/qwen7b-cube-lora --setup lora-v4.2-7b-nohint --out runs/test_nohint_lora_7b.jsonl",
    # prompt-offloading arms
    "python scripts/make_sft.py --data training_data --profile lean --out data/sft_lean",
    "python scripts/synth_schema_qa.py --db formula_1 --n 800 --db student_club --n 500 --out data/synth/synth.jsonl --audit",
    "python scripts/make_sft.py --data training_data --profile noschema --synthetic data/synth/synth.jsonl --out data/sft_noschema",
    "python -m harness.evaluate --db formula_1 --db student_club --ids-file splits/test_ids.json --no-evidence --profile lean --model ollama_chat/qwen7b-cube-lean --setup lora-lean-7b-nohint --out runs/test_nohint_lean_7b.jsonl",
]
for s in STEPS:
    print("$", s)
    if RUN_PIPELINE:
        subprocess.run(s.split("#")[0], shell=True, cwd=ROOT, check=True)

After a new run, gzip the output into `results/runs/` under the file name used in `report/analysis.py` and re-run this notebook.